# SmartContainer Risk Engine — Pipeline Notebook
## For Hackathon Judges

This notebook demonstrates the full machine learning pipeline behind the **SmartContainer Risk Engine**:
- Data loading & cleaning
- Feature engineering (discrepancy, behavioural, target encoding)
- Anomaly score injection (IsolationForest)
- Big Three Ensemble training (XGBoost + LightGBM + CatBoost)
- Ensemble feature importance

> **Note:** Correlation analysis cells are left as empty placeholders — the author will add graphs manually.

## Cell 1 — Import Required Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os
import numpy as np
import pandas as pd

# Encoders
from category_encoders import TargetEncoder

# Sklearn
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print("All libraries imported successfully.")
print(f"  numpy   : {np.__version__}")
print(f"  pandas  : {pd.__version__}")

---
## Cell 2 — Load Historical Data

Load `Historical Data.csv` (54,000 rows × 16 columns).  
All preprocessing stats (medians, frequencies, encodings) are **fitted exclusively on this dataset** — never on the test set.

**Target mapping:**
| `Clearance_Status` | Integer label |
|---|---|
| Clear | 0 (Low) |
| Low Risk | 1 (Medium) |
| Critical | 2 (Critical) |

In [ ]:
# ── Paths (adjust if running from a different working directory) ──────────
BASE_DIR   = os.path.dirname(os.path.abspath("__file__"))
TRAIN_PATH = os.path.join(BASE_DIR, "data", "Historical Data.csv")
TEST_PATH  = os.path.join(BASE_DIR, "data", "Real-Time Data.csv")

TARGET_MAP = {"Clear": 0, "Low Risk": 1, "Critical": 2}
RISK_LABELS = {0: "Low", 1: "Medium", 2: "Critical"}

NUMERICAL_COLS   = ["Declared_Value", "Declared_Weight", "Measured_Weight", "Dwell_Time_Hours"]
CATEGORICAL_COLS = [
    "Trade_Regime (Import / Export / Transit)", "Origin_Country",
    "Destination_Port", "Destination_Country", "Importer_ID", "Exporter_ID", "Shipping_Line",
]
SMOOTH_M = 10

# ── Load data ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"\nTarget distribution (train):")
print(train_df["Clearance_Status"].value_counts())

print(f"\nNull counts (train):")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

train_df.head(3)

## Cell 3 — Data Cleaning & Discrepancy Feature Engineering

Before modelling we apply two cleaning passes and engineer **three discrepancy features** that capture value/weight anomalies:

| Feature | Formula | Intuition |
|---|---|---|
| `Log_Declared_Value` | $\ln(1 + \text{Declared\_Value})$ | Compresses long-tailed value distribution |
| `Log_Value_to_Weight_Ratio` | $\ln\!\left(1 + \dfrac{\text{Declared\_Value}}{\text{Declared\_Weight}+1}\right)$ | Value-per-kg — spikes may indicate mis-declaration |
| `Weight_Diff_Ratio` | $\dfrac{\lvert\text{Measured\_Weight} - \text{Declared\_Weight}\rvert}{\text{Declared\_Weight}+1}$ | Absolute relative discrepancy between declared and measured weight |

**Cleaning steps:**
1. Numerical columns — fill nulls with **port-group median** (group = `Destination_Port`)
2. Categorical columns — fill nulls with `"Unknown"`
3. `Declaration_Date` → parse to datetime → extract `Declaration_DayOfWeek` (0–6) and `Declaration_Hour` (0–23)

In [ ]:
def _clean(df: pd.DataFrame) -> pd.DataFrame:
    """Port-group median imputation + date/time parsing."""
    df = df.copy()
    # Numerical: fill with port-group median, then global median
    for col in NUMERICAL_COLS:
        if col in df.columns:
            medians = df.groupby("Destination_Port")[col].transform("median")
            df[col] = df[col].fillna(medians).fillna(df[col].median())
    # Categorical: fill with "Unknown"
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
    # Date/time features
    if "Declaration_Date" in df.columns:
        dt = pd.to_datetime(df["Declaration_Date"], errors="coerce")
        df["Declaration_DayOfWeek"] = dt.dt.dayofweek.fillna(0).astype(int)
        df["Declaration_Hour"]      = dt.dt.hour.fillna(0).astype(int)
    return df


def _engineer_discrepancy(df: pd.DataFrame) -> pd.DataFrame:
    """Log-scaled value/weight discrepancy features."""
    df = df.copy()
    df["Log_Declared_Value"] = np.log1p(df["Declared_Value"].clip(lower=0))
    df["Log_Value_to_Weight_Ratio"] = np.log1p(
        df["Declared_Value"].clip(lower=0) / (df["Declared_Weight"].clip(lower=0) + 1)
    )
    df["Weight_Diff_Ratio"] = (
        (df["Measured_Weight"] - df["Declared_Weight"]).abs()
        / (df["Declared_Weight"].clip(lower=0) + 1)
    )
    return df


# ── Apply to both splits ──────────────────────────────────────────────────
train_clean = _engineer_discrepancy(_clean(train_df))
test_clean  = _engineer_discrepancy(_clean(test_df))

print("Discrepancy features (train sample):")
train_clean[["Log_Declared_Value", "Log_Value_to_Weight_Ratio", "Weight_Diff_Ratio"]].describe()

## 📊 [PLACEHOLDER] Correlation Analysis — Discrepancy Features

> **Graph slot:** Insert a correlation heatmap or distribution plot comparing
> `Log_Declared_Value`, `Log_Value_to_Weight_Ratio`, `Weight_Diff_Ratio` against `Clearance_Status` here.

In [ ]:
# INSERT: Correlation heatmap / distribution plots for discrepancy features here
# Example:
#   import matplotlib.pyplot as plt, seaborn as sns
#   corr_cols = ["Log_Declared_Value", "Log_Value_to_Weight_Ratio", "Weight_Diff_Ratio"]
#   sns.heatmap(train_clean[corr_cols + ["label"]].corr(), annot=True, cmap="coolwarm")
#   plt.show()
pass

## Cell 4 — Behavioural & Frequency Features

These features capture **trading-pattern risk signals** derived from historical behaviour:

| Feature | Description |
|---|---|
| `Route_ID` | Concatenation of `Origin_Country` + `Destination_Port` |
| `HS_Category` | First 2 digits of HS code (commodity group) |
| `Importer_Freq_Count` | How often this importer appears in training data |
| `Rare_Route_Flag` | 1 if route appears fewer than 5 times (uncommon corridor) |
| `Shipping_Line_Avg_Dwell` | Mean dwell time for this shipping line (train-fitted) |
| `Dwell_Time_Deviation` | $\text{Dwell\_Time\_Hours} - \text{Shipping\_Line\_Avg\_Dwell}$ |

> **No leakage:** All frequency/average maps are fitted on **training data only**, then applied to both train and test.

In [ ]:
RARE_ROUTE_THRESHOLD = 5

def _engineer_behavioural(train: pd.DataFrame, test: pd.DataFrame):
    """Frequency / behavioural features — all maps fitted on train only."""
    train, test = train.copy(), test.copy()

    # Route ID
    for df in (train, test):
        df["Route_ID"] = df["Origin_Country"].astype(str) + "_" + df["Destination_Port"].astype(str)

    # HS Category (first 2 chars of HS_Code if present)
    hs_col = next((c for c in train.columns if "hs" in c.lower()), None)
    if hs_col:
        for df in (train, test):
            df["HS_Category"] = df[hs_col].astype(str).str[:2]
    else:
        for df in (train, test):
            df["HS_Category"] = "00"

    # Importer frequency count (train-fitted)
    imp_freq = train["Importer_ID"].value_counts()
    for df in (train, test):
        df["Importer_Freq_Count"] = df["Importer_ID"].map(imp_freq).fillna(0).astype(int)

    # Rare route flag (train-fitted)
    route_freq = train["Route_ID"].value_counts()
    for df in (train, test):
        df["Rare_Route_Flag"] = (
            df["Route_ID"].map(route_freq).fillna(0) < RARE_ROUTE_THRESHOLD
        ).astype(int)

    # Shipping line average dwell (train-fitted)
    sl_avg_dwell = train.groupby("Shipping_Line")["Dwell_Time_Hours"].mean()
    for df in (train, test):
        df["Shipping_Line_Avg_Dwell"] = (
            df["Shipping_Line"].map(sl_avg_dwell)
            .fillna(train["Dwell_Time_Hours"].mean())
        )

    # Dwell time deviation
    for df in (train, test):
        df["Dwell_Time_Deviation"] = df["Dwell_Time_Hours"] - df["Shipping_Line_Avg_Dwell"]

    return train, test


train_beh, test_beh = _engineer_behavioural(train_clean, test_clean)

print("Behavioural features (train sample):")
cols_show = ["Importer_Freq_Count", "Rare_Route_Flag", "Shipping_Line_Avg_Dwell", "Dwell_Time_Deviation"]
train_beh[cols_show].describe()

## Cell 5 — Smoothed Target Encoding + Recovered Features

### Smoothed Target Encoding (anti-leakage)

For high-cardinality columns (`Importer_ID`, `HS_Category`) we use **additive smoothing**:

$$\text{encoded}(x) = \frac{n_x \cdot \bar{y}_x + M \cdot \bar{y}_{\text{global}}}{n_x + M}$$

where $n_x$ = count of category $x$, $\bar{y}_x$ = per-category target mean, $M = 10$ (smoothing factor), $\bar{y}_{\text{global}}$ = overall target mean.

This shrinks sparse categories to the global mean, reducing overfitting on rare importers.

### Recovered Features

| Feature | Method |
|---|---|
| `Trade_Regime_*` | One-hot encoding of `Trade_Regime (Import / Export / Transit)` |
| `Origin_Country_Risk` | Mean target encoding of `Origin_Country` (TargetEncoder) |
| `Exporter_Risk` | Mean target encoding of `Exporter_ID` (TargetEncoder) |

In [ ]:
def _engineer_smoothed_target_encoding(
    train: pd.DataFrame, test: pd.DataFrame, y_train: pd.Series
):
    """Additive-smoothed target encoding for Importer_ID and HS_Category."""
    train, test = train.copy(), test.copy()
    global_mean = y_train.mean()

    for col, out_col in [("Importer_ID", "Importer_Risk_Index"), ("HS_Category", "HS_Risk_Index")]:
        stats = (
            pd.concat([train[[col]], y_train.rename("label")], axis=1)
            .groupby(col)["label"]
            .agg(["mean", "count"])
        )
        stats["smoothed"] = (
            (stats["count"] * stats["mean"] + SMOOTH_M * global_mean)
            / (stats["count"] + SMOOTH_M)
        )
        enc_map = stats["smoothed"].to_dict()
        train[out_col] = train[col].map(enc_map).fillna(global_mean)
        test[out_col]  = test[col].map(enc_map).fillna(global_mean)

    return train, test


def _engineer_recovered_features(
    train: pd.DataFrame, test: pd.DataFrame, y_train: pd.Series
):
    """One-hot Trade_Regime + TargetEncoder for Origin_Country and Exporter_ID."""
    train, test = train.copy(), test.copy()
    regime_col = "Trade_Regime (Import / Export / Transit)"

    # One-hot encode Trade_Regime
    if regime_col in train.columns:
        dummies_train = pd.get_dummies(train[regime_col], prefix="Trade_Regime")
        dummies_test  = pd.get_dummies(test[regime_col],  prefix="Trade_Regime")
        dummies_test  = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
        train = pd.concat([train, dummies_train], axis=1)
        test  = pd.concat([test,  dummies_test],  axis=1)

    # TargetEncoder for Origin_Country and Exporter_ID
    te = TargetEncoder(cols=["Origin_Country", "Exporter_ID"])
    te.fit(train[["Origin_Country", "Exporter_ID"]], y_train)
    enc_train = te.transform(train[["Origin_Country", "Exporter_ID"]])
    enc_test  = te.transform(test[["Origin_Country",  "Exporter_ID"]])
    train["Origin_Country_Risk"] = enc_train["Origin_Country"].values
    train["Exporter_Risk"]       = enc_train["Exporter_ID"].values
    test["Origin_Country_Risk"]  = enc_test["Origin_Country"].values
    test["Exporter_Risk"]        = enc_test["Exporter_ID"].values

    return train, test


# ── Encode ────────────────────────────────────────────────────────────────
y_train_raw = train_beh["Clearance_Status"].map(TARGET_MAP)

train_enc, test_enc = _engineer_smoothed_target_encoding(train_beh, test_beh, y_train_raw)
train_enc, test_enc = _engineer_recovered_features(train_enc, test_enc, y_train_raw)

# ── Select final feature columns ─────────────────────────────────────────
ENGINEERED_FEATURES = [
    "Declaration_DayOfWeek", "Declaration_Hour",
    "Dwell_Time_Deviation", "Exporter_Risk", "HS_Risk_Index",
    "Importer_Freq_Count", "Importer_Risk_Index", "Log_Declared_Value",
    "Log_Value_to_Weight_Ratio", "Origin_Country_Risk", "Rare_Route_Flag",
    "Shipping_Line_Avg_Dwell", "Weight_Diff_Ratio",
]
TRADE_DUMMIES = [c for c in train_enc.columns if c.startswith("Trade_Regime_")]
ALL_FEATURES  = ENGINEERED_FEATURES + TRADE_DUMMIES

common_cols = [c for c in ALL_FEATURES if c in train_enc.columns and c in test_enc.columns]

X_train = train_enc[common_cols].reset_index(drop=True)
X_test  = test_enc[common_cols].reset_index(drop=True)
y_train = y_train_raw.reset_index(drop=True)

print(f"X_train : {X_train.shape},  X_test : {X_test.shape}")
print(f"Features : {common_cols}")
X_train.head(3)

## 📊 [PLACEHOLDER] Correlation Analysis — Encoded Features

> **Graph slot:** Insert your correlation / distribution plots here.
>
> Suggested visuals:
> - Bar chart of `Importer_Risk_Index` grouped by `Clearance_Status`
> - Distribution of `Origin_Country_Risk` split by `Clearance_Status`
> - Correlation heatmap of all engineered features vs numeric target

In [ ]:
# INSERT: Your correlation / distribution plots for encoded features here.
pass

## Cell 6 — IsolationForest Anomaly Score Injection

An **IsolationForest** (contamination = 5%) is fitted on the full training feature matrix.
Raw decision scores are **normalised to 0–100** using training-set min/max, then appended as `Anomaly_Score` to both splits.

The fitted `iso`, `iso_rmin`, and `iso_rmax` are retained so production inference applies the **same normalisation** without re-fitting — no leakage.

> Containers scoring close to 100 are structural outliers — atypical on multiple feature dimensions simultaneously.

In [ ]:
IF_CONTAMINATION = 0.05
RANDOM_STATE     = 42

iso = IsolationForest(
    contamination=IF_CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_train)

raw_tr = -iso.decision_function(X_train)
raw_te = -iso.decision_function(X_test)

iso_rmin = float(raw_tr.min())
iso_rmax = float(raw_tr.max())
_denom   = iso_rmax - iso_rmin if iso_rmax != iso_rmin else 1.0

X_train = X_train.copy()
X_test  = X_test.copy()
X_train["Anomaly_Score"] = np.clip((raw_tr - iso_rmin) / _denom * 100, 0, 100)
X_test["Anomaly_Score"]  = np.clip((raw_te - iso_rmin) / _denom * 100, 0, 100)

print(f"IsolationForest fitted — iso_rmin={iso_rmin:.4f}  iso_rmax={iso_rmax:.4f}")
print(f"Anomaly_Score (train) : mean={X_train['Anomaly_Score'].mean():.2f}  max={X_train['Anomaly_Score'].max():.2f}")
print(f"Anomaly_Score (test)  : mean={X_test['Anomaly_Score'].mean():.2f}  max={X_test['Anomaly_Score'].max():.2f}")
print(f"\nFinal X_train : {X_train.shape}  |  X_test : {X_test.shape}")

## Cell 7 — Hyperparameter Optimisation (Optuna × XGBoost)

A **20-trial TPE study** maximises macro-F1 via 3-fold stratified cross-validation.

`IsolationForest` is **re-fitted inside each CV fold** so anomaly scores never leak from validation data into training.

**Tuned parameters:** `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_TRIALS = 20
CV_FOLDS      = 3

# Sample weights for cost-sensitive training
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
print(f"Sample weights — min: {sample_weights.min():.4f}  max: {sample_weights.max():.4f}")


def _inject_anomaly_cv(Xtr, Xval):
    """Re-fit IsolationForest per fold to prevent anomaly-score leakage."""
    iso_cv = IsolationForest(contamination=IF_CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
    iso_cv.fit(Xtr)
    raw_tr  = -iso_cv.decision_function(Xtr)
    raw_val = -iso_cv.decision_function(Xval)
    rmin, rmax = raw_tr.min(), raw_tr.max()
    d = rmax - rmin if rmax != rmin else 1.0
    Xtr  = Xtr.copy();  Xtr["Anomaly_Score"]  = np.clip((raw_tr  - rmin) / d * 100, 0, 100)
    Xval = Xval.copy(); Xval["Anomaly_Score"] = np.clip((raw_val - rmin) / d * 100, 0, 100)
    return Xtr, Xval


def objective(trial):
    params = {
        "objective":             "multi:softprob",
        "num_class":             3,
        "eval_metric":           "mlogloss",
        "use_label_encoder":     False,
        "tree_method":           "hist",
        "random_state":          RANDOM_STATE,
        "n_estimators":          500,
        "early_stopping_rounds": 30,
        "max_depth":             trial.suggest_int("max_depth", 3, 9),
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    skf    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        Xtr_raw, Xval_raw = X_train.iloc[train_idx], X_train.iloc[val_idx]
        ytr,  yval        = y_train.iloc[train_idx], y_train.iloc[val_idx]
        wtr               = sample_weights[train_idx]
        Xtr, Xval = _inject_anomaly_cv(Xtr_raw, Xval_raw)
        clf = XGBClassifier(**params)
        clf.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xval, yval)], verbose=False)
        scores.append(f1_score(yval, clf.predict(Xval), average="macro"))
    return np.mean(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f"\nBest macro-F1 (CV): {study.best_value:.4f}")
print(f"Best params:        {best_params}")

## Cell 8 — Train Big Three Ensemble

| Model | Configuration | Blend weight |
|---|---|---|
| **XGBoost** | Optuna best params · `n_estimators=800` · cost-sensitive `sample_weight` | **0.5** |
| **LightGBM** | `n_estimators=800` · `class_weight="balanced"` | **0.3** |
| **CatBoost** | `iterations=800` · `auto_class_weights="Balanced"` | **0.2** |

Blended probability: $P = 0.5 \cdot P_{\text{XGB}} + 0.3 \cdot P_{\text{LGB}} + 0.2 \cdot P_{\text{CAT}}$

In [ ]:
# ── XGBoost (Optuna-tuned) ────────────────────────────────────────────────
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    use_label_encoder=False,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_estimators=800,
    **best_params,
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights, verbose=False)
print(f"XGBoost  trained — {X_train.shape}")

# ── LightGBM ──────────────────────────────────────────────────────────────
lgb_model = LGBMClassifier(
    n_estimators=800,
    num_class=3,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    verbose=-1,
)
lgb_model.fit(X_train, y_train)
print("LightGBM trained")

# ── CatBoost ──────────────────────────────────────────────────────────────
cat_model = CatBoostClassifier(
    iterations=800,
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    verbose=False,
)
cat_model.fit(X_train, y_train)
print("CatBoost trained")
print("\nAll three models trained successfully.")

## Cell 9 — Ensemble Inference with Custom Thresholds

Thresholds are applied to **blended probabilities** (not argmax) for fine-grained control over `Critical` class recall:

| Rule | Label |
|---|---|
| $P(\text{Critical}) > 0.30$ | → `Critical` (class 2) |
| $P(\text{Medium}) > 0.40$ and not Critical | → `Medium` (class 1) |
| Otherwise | → `Low` (class 0) |

**Continuous Risk Score (0–100):** Rank-based percentile within each tier ensures uniform spread:
- Low → 0–33 · Medium → 34–66 · Critical → 67–100

In [ ]:
CRITICAL_THRESHOLD = 0.30
MEDIUM_THRESHOLD   = 0.40

# ── Blend probabilities ───────────────────────────────────────────────────
xgb_proba = xgb_model.predict_proba(X_test)
lgb_proba = lgb_model.predict_proba(X_test)
cat_proba = cat_model.predict_proba(X_test)
proba = (0.5 * xgb_proba) + (0.3 * lgb_proba) + (0.2 * cat_proba)

# ── Custom threshold classification ───────────────────────────────────────
predictions   = np.zeros(len(X_test), dtype=int)
critical_mask = proba[:, 2] > CRITICAL_THRESHOLD
medium_mask   = (~critical_mask) & (proba[:, 1] > MEDIUM_THRESHOLD)
predictions[critical_mask] = 2
predictions[medium_mask]   = 1

# ── Rank-based risk score (uniform spread within each tier) ───────────────
raw_score = (proba[:, 1] * 50) + (proba[:, 2] * 100)
TIERS     = {0: (0.0, 33.0), 1: (34.0, 66.0), 2: (67.0, 100.0)}
risk_scores = np.empty(len(predictions), dtype=float)
for cls, (lo, hi) in TIERS.items():
    mask = predictions == cls
    if not mask.any():
        continue
    vals = raw_score[mask]
    n    = mask.sum()
    risk_scores[mask] = (
        (lo + hi) / 2 if n == 1
        else lo + vals.argsort().argsort() / (n - 1) * (hi - lo)
    )

print(f"Predictions — Low: {(predictions==0).sum()}  "
      f"Medium: {(predictions==1).sum()}  "
      f"Critical: {(predictions==2).sum()}")

# ── Train-set sanity check ────────────────────────────────────────────────
train_proba = (
    (0.5 * xgb_model.predict_proba(X_train))
    + (0.3 * lgb_model.predict_proba(X_train))
    + (0.2 * cat_model.predict_proba(X_train))
)
train_preds = np.argmax(train_proba, axis=1)
print("\nClassification report on X_train (sanity check):")
print(classification_report(y_train, train_preds, target_names=["Low", "Medium", "Critical"]))

## Cell 10 — Ensemble Feature Importance

Normalised feature importances from all three models are averaged using the same blend weights (0.5 / 0.3 / 0.2).

In [ ]:
feature_names = X_train.columns.tolist()

def _normed(arr):
    s = arr.sum()
    return arr / s if s > 0 else arr

xgb_norm = _normed(xgb_model.feature_importances_)
lgb_norm = _normed(lgb_model.feature_importances_)
cat_norm = _normed(cat_model.feature_importances_)
blended  = (0.5 * xgb_norm) + (0.3 * lgb_norm) + (0.2 * cat_norm)

importance_df = (
    pd.DataFrame({
        "Feature":                feature_names,
        "XGBoost":                xgb_norm,
        "LightGBM":               lgb_norm,
        "CatBoost":               cat_norm,
        "Blended (0.5/0.3/0.2)":  blended,
    })
    .sort_values("Blended (0.5/0.3/0.2)", ascending=False)
    .reset_index(drop=True)
)

print("Ensemble Feature Importance (top features first):")
importance_df

## 📊 [PLACEHOLDER] Feature Importance Chart

> **Graph slot:** Insert your feature importance bar chart here.
>
> Suggested: horizontal bar chart of `Blended (0.5/0.3/0.2)` from `importance_df`

In [ ]:
# INSERT: Your feature importance visualisation here.
pass

## Cell 11 — Export Predictions

In [ ]:
test_ids = test_enc["Container_ID"].reset_index(drop=True) if "Container_ID" in test_enc.columns else pd.RangeIndex(len(X_test))

output_df = pd.DataFrame({
    "Container_ID":  test_ids,
    "Risk_Label":    pd.Series(predictions).map(RISK_LABELS),
    "Risk_Score":    np.round(risk_scores, 2),
    "P_Low":         np.round(proba[:, 0], 4),
    "P_Medium":      np.round(proba[:, 1], 4),
    "P_Critical":    np.round(proba[:, 2], 4),
    "Anomaly_Score": np.round(X_test["Anomaly_Score"].values, 2),
})

print(output_df["Risk_Label"].value_counts())
print(f"\nRisk Score — mean: {output_df['Risk_Score'].mean():.2f}  max: {output_df['Risk_Score'].max():.2f}")
output_df.head(10)